In [ ]:
import os
from pathlib import Path

os.environ["OPENAI_API_KEY"] = "your-api-key-here"

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_folder = Path.home() / "Downloads" / "nlp rag"
pdf_files = list(pdf_folder.rglob("*.pdf"))

docs = []

for pdf_file in pdf_files:
    loader = PyPDFLoader(str(pdf_file))
    docs.extend(loader.load())

print(f"Loaded {len(docs)} pages from {len(pdf_files)} PDF documents")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(docs)

print(f"Created {len(chunks)} text chunks")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(chunks, embeddings)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

prompt = ChatPromptTemplate.from_template("""
Answer the question using only the provided document context.

Context:
{context}

Question:
{question}
""")

chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

question = "Compare the key changes in financial performance across the provided financial reports."

answer = chain.invoke(question)

print(answer)